# Topic 3. 출시 시점-구매 시점 차이 → 리뷰 라이프사이클 분석

> 데이터: `si_dataset/review_for_analysis.json`  
> 독립 실행: `uv run jupyter notebook topic3_analysis.ipynb`

**분석 방법**: 코호트 분석, 평점 시계열, Mann-Whitney U, BERTopic topics_over_time, KeyBERT, UMAP 2D

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter
from pathlib import Path

# sns.set_theme 먼저 — 이후에 설정한 폰트를 덮어쓰지 않도록
sns.set_theme(style='whitegrid', palette='muted')

# 한글 폰트 — set_theme 이후에 지정해야 Arial로 리셋되지 않음
_font_path = r'C:\Windows\Fonts\malgun.ttf'
fm.fontManager.addfont(_font_path)
_font_name = fm.FontProperties(fname=_font_path).get_name()
plt.rcParams['font.family'] = _font_name
plt.rcParams['axes.unicode_minus'] = False
# seaborn rc도 같이 덮어쓰기
sns.set_theme(style='whitegrid', palette='muted', rc={
    'font.family': _font_name,
    'axes.unicode_minus': False,
})

print(f'사용 폰트: {plt.rcParams["font.family"]}')

DATA_PATH = Path('si_dataset/review_for_analysis.json')

In [ ]:
# JSON에 NaN, nullDeliberate 등 비표준 값이 포함되어 있어 전처리 후 로드
raw = DATA_PATH.read_text(encoding='utf-8')
raw = re.sub(r'\bNaN\b', 'null', raw)
raw = re.sub(r'\bnullDeliberate\b', 'null', raw)
records = json.loads(raw)

df = pd.json_normalize(records)
print(f'총 레코드: {len(df):,}개')
print(f'컬럼 수: {df.shape[1]}개')
df.head(2)

In [ ]:
from scipy import stats

# ── 재현성 시드 고정 ──
import random
SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)

# 분석에 쓸 핵심 컬럼만 정리
df['reviewAt_dt'] = pd.to_datetime(df['reviewAt'], unit='ms')
df['content_len'] = df['content'].fillna('').str.len()
df['brand'] = df['product_name'].apply(
    lambda x: 'Apple' if 'iphone' in str(x) else 'Samsung'
)

# 제품명 짧게
product_labels = {
    'iphone_17': 'iPhone 17',
    'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26',
    'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7',
    'galaxy_z_flip7': 'Galaxy Z Flip7',
}
df['product_label'] = df['product_name'].map(product_labels)

print(df[['product_name', 'brand', 'rating', 'helpfulCount', 'content_len', 'reviewAt_dt']].dtypes)
df[['product_label', 'brand', 'rating', 'helpfulCount', 'helpfulTrueCount', 'helpfulFalseCount', 'content_len']].describe()

---
### 2-9. 형태소 분석 기반 TF-IDF (Kiwi) — 워드클라우드 고도화

> 공백 분리의 한계를 극복: 명사(NNG/NNP) + 형용사(VA) 단위로 정밀 추출

In [ ]:
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords

kiwi = Kiwi()

# 분석에 쓸 품사 (명사류 + 형용사)
KEEP_POS = {'NNG', 'NNP', 'SL', 'VA'}  # 일반명사, 고유명사, 외래어, 형용사

# 도메인·제품명 등 분석에 무의미한 토큰
CUSTOM_STOP = {
    '아이폰', '갤럭시', '폰', '핸드폰', '스마트폰', '제품', '구매', '배송',
    '리뷰', '후기', '구입', '사용', '이것', '저것', '그것', '하나',
    '생각', '정말', '진짜', '너무', '이번', '것', '수', '때', '제',
    '분', '번', '개', '명', '원', '년', '월', '일',
}

def extract_morphemes(text: str) -> list[str]:
    """Kiwi로 형태소 분석 → 명사/형용사 추출"""
    if not isinstance(text, str) or len(text.strip()) < 2:
        return []
    tokens = []
    for sent in kiwi.analyze(text[:2000]):  # 너무 긴 텍스트는 2000자로 제한
        for tok in sent[0]:
            form = tok.form.strip()
            tag = str(tok.tag)
            if tag in KEEP_POS and len(form) >= 2 and form not in CUSTOM_STOP:
                tokens.append(form)
        break  # 첫 번째 분석 결과만 사용
    return tokens

# ── 전체 리뷰 형태소 분석 (캐싱) ────────────────────────────
import pickle, os

CACHE_PATH = 'output/morpheme_cache.pkl'
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        df_tokens = pickle.load(f)
    print(f'캐시 로드 완료: {len(df_tokens)}건')
else:
    print('형태소 분석 시작... (1~2분 소요)')
    from tqdm import tqdm
    tqdm.pandas()
    df_tokens = df['content'].fillna('').progress_apply(extract_morphemes)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(df_tokens, f)
    print('분석 완료 & 캐시 저장')

df['tokens'] = df_tokens
df['token_str'] = df['tokens'].apply(lambda t: ' '.join(t))

# 샘플 확인
print('\n샘플 형태소 분석:')
for _, row in df[df['content_len'] > 100].sample(2, random_state=SEED).iterrows():
    print(f'  원문: {str(row["content"])[:80]}...')
    print(f'  토큰: {row["tokens"][:15]}\n')

---
---
# Topic 3. 출시 시점-구매 시점 차이 → 리뷰 라이프사이클 분석
## "같은 폰, 다른 시간: 얼리어답터와 일반 구매자는 같은 폰을 다르게 경험하는가?"

**핵심 질문**
1. 출시 직후 구매자(얼리어답터)와 수개월 후 구매자(일반 소비자)의 평점이 다른가?
2. 리뷰의 관심사·키워드는 출시 후 시간이 지남에 따라 어떻게 바뀌는가?
3. 초기 리뷰가 플랫폼 전체 평점을 왜곡하는가? (허니문 효과)
4. 출시 초기 리뷰가 더 많은 공감을 받는가?

**코호트 정의**

| 코호트 | 기준 | 소비자 유형 |
|--------|------|-----------|
| D+0~30 | 출시 후 30일 이내 | 얼리어답터 — 브랜드 충성, 높은 관여도 |
| D+31~90 | 출시 31~90일 | 초기 다수 — 입소문 듣고 구매 |
| D+91+ | 출시 91일 이후 | 후기 다수 — 가격 하락 후 구매, 장기 사용자 |

셀 1~4 + Topic 2 셀이 먼저 실행되어 있어야 합니다 (`df`, `kiwi`, `SEED` 등 변수 필요).

In [ ]:
# ── 출시일 (첫 리뷰 날짜로 검증된 수치) ─────────────────────
RELEASE_DATES_T3 = {
    'iphone_17':          pd.Timestamp('2025-09-19'),
    'iphone_17_pro':      pd.Timestamp('2025-09-19'),
    'iphone_17_pro_max':  pd.Timestamp('2025-09-19'),
    'galaxy_s26':         pd.Timestamp('2026-02-22'),  # 첫 리뷰 03-04 → ~10일 전
    'galaxy_s26_ultra':   pd.Timestamp('2026-02-22'),
    'galaxy_z_fold7':     pd.Timestamp('2025-07-10'),  # 첫 리뷰 07-20 → ~10일 전
    'galaxy_z_flip7':     pd.Timestamp('2025-07-10'),
}

PRODUCT_LABELS = {
    'iphone_17': 'iPhone 17', 'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26', 'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7', 'galaxy_z_flip7': 'Galaxy Z Flip7',
}
BRAND_COLORS_T3 = {
    'iPhone 17': '#4A90D9', 'iPhone 17 Pro': '#2C5F8A', 'iPhone 17 Pro Max': '#1A3A55',
    'Galaxy S26': '#E85D5D', 'Galaxy S26 Ultra': '#A52020',
    'Galaxy Z Fold7': '#E8943A', 'Galaxy Z Flip7': '#B85E10',
}
COHORT_ORDER  = ['D+0~30 (얼리어답터)', 'D+31~90 (초기다수)', 'D+91+ (후기다수)']
COHORT_COLORS = {'D+0~30 (얼리어답터)': '#2196F3',
                 'D+31~90 (초기다수)':  '#FF9800',
                 'D+91+ (후기다수)':    '#9C27B0'}

# ── days_since_release 재계산 (수정된 출시일 기준) ────────────
df['release_date_t3'] = df['product_name'].map(RELEASE_DATES_T3)
df['days_t3'] = (df['reviewAt_dt'] - df['release_date_t3']).dt.days

def assign_cohort(d):
    if pd.isna(d) or d < 0: return '출시 전'
    if d <= 30:  return 'D+0~30 (얼리어답터)'
    if d <= 90:  return 'D+31~90 (초기다수)'
    return 'D+91+ (후기다수)'

df['cohort'] = df['days_t3'].apply(assign_cohort)

# 코호트 분포 확인
print('코호트 분포:')
cohort_dist = df[df['cohort'] != '출시 전'].groupby(['product_label', 'cohort']).size().unstack(fill_value=0)
cohort_dist = cohort_dist.reindex(columns=[c for c in COHORT_ORDER if c in cohort_dist.columns])
print(cohort_dist.to_string())
print(f'\n전체: {(df["cohort"]!="출시 전").sum()}건 / 출시 전: {(df["cohort"]=="출시 전").sum()}건')

### 3-1. 리뷰 볼륨 타임라인 — 언제 리뷰가 폭발하는가

In [ ]:
df_valid = df[df['cohort'] != '출시 전'].copy()

# 10일 bin으로 집계
df_valid['days_bin'] = (df_valid['days_t3'] // 10) * 10

products = list(PRODUCT_LABELS.values())
# Apple vs Samsung 나눠서 2행 4열
apple_prods = [p for p in products if 'iPhone' in p]
sam_prods   = [p for p in products if 'Galaxy' in p]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax_idx, prod in enumerate(apple_prods + sam_prods):
    ax = axes[ax_idx]
    pname = [k for k, v in PRODUCT_LABELS.items() if v == prod][0]
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty:
        ax.axis('off'); continue

    vol = sub.groupby('days_bin').size().reset_index(name='count')
    color = BRAND_COLORS_T3[prod]

    bars = ax.bar(vol['days_bin'], vol['count'], width=9,
                  color=color, alpha=0.8, edgecolor='white')
    # 최고 스파이크 강조
    peak_day = vol.loc[vol['count'].idxmax(), 'days_bin']
    peak_cnt = vol['count'].max()
    ax.axvline(peak_day, color='red', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.text(peak_day + 1, peak_cnt * 0.95,
            f'Peak\nD+{int(peak_day)}~{int(peak_day)+9}', fontsize=7, color='red')

    # 코호트 경계선
    for boundary, label in [(30, 'D+30'), (90, 'D+90')]:
        if boundary <= sub['days_t3'].max():
            ax.axvline(boundary, color='gray', linestyle=':', linewidth=1, alpha=0.5)
            ax.text(boundary + 1, ax.get_ylim()[1] * 0.85, label, fontsize=7, color='gray')

    ax.set_title(prod, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('출시 후 경과 일수 (10일 bin)')
    ax.set_ylabel('리뷰 수')

for ax in axes[len(apple_prods + sam_prods):]:
    ax.axis('off')

plt.suptitle('제품별 리뷰 볼륨 타임라인 (출시 후 경과 일수 기준)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_01_volume_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# 요약
print('제품별 리뷰 볼륨 피크 시점:')
for prod in apple_prods + sam_prods:
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty: continue
    vol = sub.groupby('days_bin').size()
    peak = vol.idxmax()
    total_days = sub['days_t3'].max()
    print(f'  {prod:<22} 피크: D+{int(peak)}~{int(peak)+9}일  '
          f'(전체 관찰기간: D+0~{int(total_days)})')

### 3-2. 평점 시계열 + 허니문 효과 검증

In [ ]:
# ── 평점 시계열 (10일 bin 이동평균) ───────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax_idx, prod in enumerate(apple_prods + sam_prods):
    ax = axes[ax_idx]
    sub = df_valid[df_valid['product_label'] == prod]
    if sub.empty: ax.axis('off'); continue

    color = BRAND_COLORS_T3[prod]
    rating_by_bin = sub.groupby('days_bin')['rating'].agg(['mean', 'count', 'sem']).reset_index()
    rating_by_bin = rating_by_bin[rating_by_bin['count'] >= 3]

    # 산점도 (개별 리뷰, jitter)
    jitter = np.random.normal(0, 1.5, len(sub))
    ax.scatter(sub['days_t3'] + jitter, sub['rating'],
               alpha=0.05, s=5, color=color)

    # 10일 bin 평균 + 95% CI
    ax.plot(rating_by_bin['days_bin'], rating_by_bin['mean'],
            color=color, linewidth=2.5, zorder=5)
    ax.fill_between(rating_by_bin['days_bin'],
                    rating_by_bin['mean'] - 1.96 * rating_by_bin['sem'],
                    rating_by_bin['mean'] + 1.96 * rating_by_bin['sem'],
                    alpha=0.2, color=color, label='95% CI')

    # 코호트 경계
    for b in [30, 90]:
        if b < sub['days_t3'].max():
            ax.axvline(b, color='gray', linestyle=':', linewidth=1)

    ax.set_ylim(1, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_title(prod, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('출시 후 경과 일수')
    ax.set_ylabel('평점')
    ax.axhline(sub['rating'].mean(), color='black', linestyle='--',
               linewidth=0.8, alpha=0.5)

for ax in axes[len(apple_prods + sam_prods):]:
    ax.axis('off')

plt.suptitle('제품별 평점 시계열 (출시 후 경과 일수 기준)\n점선=전체 평균, 회색점선=코호트 경계',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_02_rating_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 허니문 효과 검증: 코호트별 평점 + Mann-Whitney U ──────────
cohort_rating = (
    df_valid[df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby(['product_label', 'cohort'])['rating']
    .mean().unstack().reindex(columns=COHORT_ORDER)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 코호트별 평균 평점 히트맵
sns.heatmap(cohort_rating, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=3.8, vmax=5.0, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': '평균 평점'})
axes[0].set_title('제품 × 코호트 평균 평점\n(허니문 효과 = 얼리어답터 > 후기다수)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('')

# 얼리어답터 vs 후기다수 차이 막대
early_col = 'D+0~30 (얼리어답터)'
late_col  = 'D+91+ (후기다수)'
diff_df = cohort_rating[[early_col, late_col]].dropna()
diff = (diff_df[early_col] - diff_df[late_col]).sort_values()

bar_colors = ['#d9534f' if v > 0 else '#5cb85c' for v in diff.values]
bars = axes[1].barh(diff.index, diff.values, color=bar_colors, edgecolor='white', linewidth=1.2)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('얼리어답터 − 후기다수 평점 차이\n(빨강=얼리어답터가 더 후한 평가, 초록=반대)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('평점 차이')
for bar, v in zip(bars, diff.values):
    axes[1].text(v + (0.005 if v >= 0 else -0.005),
                 bar.get_y() + bar.get_height()/2,
                 f'{v:+.2f}', va='center',
                 ha='left' if v >= 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/t3_03_honeymoon.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Mann-Whitney U (얼리어답터 vs 후기다수, 제품별) ──────────
print('=== 허니문 효과 통계 검증 (얼리어답터 D+0~30 vs 후기다수 D+91+) ===')
print(f'{"제품":<22}  조기평균  후기평균  차이    U-stat   p-value  유의')
print('-' * 75)
honeymoon_products = []
for prod in apple_prods + sam_prods:
    sub = df_valid[df_valid['product_label'] == prod]
    early = sub[sub['cohort'] == early_col]['rating'].dropna()
    late  = sub[sub['cohort'] == late_col]['rating'].dropna()
    if len(early) < 5 or len(late) < 5:
        print(f'{prod:<22}  (샘플 부족)')
        continue
    u, p = stats.mannwhitneyu(early, late, alternative='two-sided')
    diff_val = early.mean() - late.mean()
    sig = '★★' if p < 0.01 else ('★' if p < 0.05 else '')
    if diff_val > 0 and p < 0.05:
        honeymoon_products.append(prod)
    print(f'{prod:<22}  {early.mean():.2f}    {late.mean():.2f}    '
          f'{diff_val:+.2f}   {u:8.0f}  {p:.4f}  {sig}')

print(f'\n허니문 효과 유의 제품: {honeymoon_products if honeymoon_products else "없음"}')

### 3-3. Helpfulness 시계열 — 초기 리뷰가 더 많은 공감을 받는가

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 코호트별 평균 helpfulness ────────────────────────────────
cohort_help = (
    df_valid[df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby('cohort')['helpfulTrueCount']
    .agg(['mean', 'median', 'count'])
    .reindex(COHORT_ORDER)
)
colors_c = [COHORT_COLORS[c] for c in COHORT_ORDER]
bars = axes[0].bar(range(len(COHORT_ORDER)), cohort_help['mean'],
                   color=colors_c, edgecolor='white', linewidth=1.2)
axes[0].set_xticks(range(len(COHORT_ORDER)))
axes[0].set_xticklabels(['얼리어답터\nD+0~30', '초기다수\nD+31~90', '후기다수\nD+91+'], fontsize=9)
axes[0].set_title('코호트별 평균 도움돼요', fontsize=12, fontweight='bold')
axes[0].set_ylabel('평균 helpfulTrueCount')
for bar, v in zip(bars, cohort_help['mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── days_since_release vs helpfulness 산점도 (전체) ──────────
ax2 = axes[1]
for prod in apple_prods[:2] + sam_prods[:2]:   # 대표 4개만
    sub = df_valid[(df_valid['product_label'] == prod) & (df_valid['helpfulTrueCount'] > 0)]
    if sub.empty: continue
    ax2.scatter(sub['days_t3'], sub['helpfulTrueCount'].clip(upper=50),
                alpha=0.3, s=10, color=BRAND_COLORS_T3[prod], label=prod)

# 전체 추세선 (binned mean)
trend = df_valid.groupby('days_bin')['helpfulTrueCount'].mean().reset_index()
ax2.plot(trend['days_bin'], trend['helpfulTrueCount'], 'k-', linewidth=2,
         label='전체 평균', zorder=10)
ax2.set_title('출시 후 경과 일수 vs 도움돼요\n(50 이상 clip)', fontsize=11, fontweight='bold')
ax2.set_xlabel('출시 후 경과 일수')
ax2.set_ylabel('helpfulTrueCount')
ax2.legend(fontsize=7, loc='upper right')

# ── 부정 리뷰의 코호트별 helpfulness (초기 부정이 더 공감받나?) ─
neg_help = (
    df_valid[df_valid['rating'] <= 2]
    [df_valid['cohort'].isin(COHORT_ORDER)]
    .groupby('cohort')['helpfulTrueCount']
    .agg(['mean', 'count'])
    .reindex(COHORT_ORDER)
)
ax3 = axes[2]
bars3 = ax3.bar(range(len(COHORT_ORDER)), neg_help['mean'],
                color=colors_c, edgecolor='white', linewidth=1.2)
ax3.set_xticks(range(len(COHORT_ORDER)))
ax3.set_xticklabels(['얼리어답터\nD+0~30', '초기다수\nD+31~90', '후기다수\nD+91+'], fontsize=9)
ax3.set_title('부정 리뷰(1-2점)의 코호트별\n평균 도움돼요', fontsize=11, fontweight='bold')
ax3.set_ylabel('평균 helpfulTrueCount')
for bar, (mean_v, cnt) in zip(bars3, neg_help.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{mean_v:.1f}\n(n={int(cnt)})', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('output/t3_04_helpfulness.png', dpi=150, bbox_inches='tight')
plt.show()

# Kruskal-Wallis
samples = [df_valid[df_valid['cohort']==c]['helpfulTrueCount'].dropna().values
           for c in COHORT_ORDER if df_valid[df_valid['cohort']==c].shape[0] > 0]
stat, p = stats.kruskal(*samples)
print(f'Helpfulness 코호트 간 Kruskal-Wallis: H={stat:.3f}, p={p:.4f}')
print('→', '코호트 간 helpfulness 차이 유의' if p < 0.05 else '차이 없음')
print('\n코호트별 통계:')
print(cohort_help.to_string())

### 3-4. BERTopic `topics_over_time` — 토픽이 시간에 따라 어떻게 변하는가
> **LDA 대신 BERTopic Dynamic Topic Modeling 사용**  
> 각 토픽이 출시 후 경과 시간에 따라 얼마나 활발하게 논의되는지 추적

In [ ]:
import warnings, pickle; warnings.filterwarnings('ignore')
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer as CV_bert2, TfidfVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

# token_str이 df에 없으면 캐시에서 직접 복구
if 'token_str' not in df.columns:
    with open('output/morpheme_cache.pkl', 'rb') as f:
        _tok = pickle.load(f)
    df['tokens']    = _tok
    df['token_str'] = df['tokens'].apply(lambda t: ' '.join(t) if isinstance(t, list) else '')
    print('token_str 복구 완료')

# 출시 후 리뷰 + 토큰 있는 것만
_mask = (df['cohort'] != '출시 전') & (df['token_str'].fillna('').str.strip().str.len() > 0)
df_tot = df[_mask].copy().reset_index(drop=True)
if 'days_bin' not in df_tot.columns:
    df_tot['days_bin'] = (df_tot['days_t3'] // 10) * 10

tot_docs = df_tot['token_str'].tolist()
print(f'BERTopic 입력 문서: {len(tot_docs)}건')

tfidf_emb2 = TfidfVectorizer(token_pattern=r'[가-힣]{2,}',
                              max_features=2000, sublinear_tf=True)
emb2 = tfidf_emb2.fit_transform(tot_docs).toarray()

umap2    = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
hdbscan2 = HDBSCAN(min_cluster_size=15, min_samples=3,
                   metric='euclidean', cluster_selection_method='eom', prediction_data=True)
cv2      = CV_bert2(ngram_range=(1, 2), min_df=1, token_pattern=r'[가-힣]{2,}')
ctfidf2  = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model_t3 = BERTopic(
    language='multilingual',
    umap_model=umap2, hdbscan_model=hdbscan2,
    vectorizer_model=cv2, ctfidf_model=ctfidf2,
    nr_topics=8, verbose=False, calculate_probabilities=False,
)
topics_t3, _ = topic_model_t3.fit_transform(tot_docs, embeddings=emb2)
df_tot['bert_topic_t3'] = topics_t3

ti3 = topic_model_t3.get_topic_info()
print(f'\n토픽 수: {(ti3["Topic"]!=-1).sum()}개  |  Outlier: {(pd.Series(topics_t3)==-1).mean()*100:.1f}%')
print('\n토픽 요약:')
for _, row in ti3[ti3['Topic']!=-1].iterrows():
    top3 = ' / '.join([w for w, _ in topic_model_t3.get_topic(row['Topic'])[:3]])
    print(f'  T{row["Topic"]:2d} (n={row["Count"]:4d}): {top3}')

In [ ]:
# ── topics_over_time: 코호트 단위로 토픽 비중 변화 추적 ───────
# BERTopic topics_over_time은 datetime을 bins로 나눔
# days_t3를 직접 timestamps로 사용 (출시 기준 상대 시간)
# 단, topics_over_time은 실제 datetime을 요구하므로 days를 anchor date로 offset

ANCHOR = pd.Timestamp('2000-01-01')
df_tot['pseudo_dt'] = ANCHOR + pd.to_timedelta(df_tot['days_t3'].clip(lower=0), unit='D')

topics_over_time = topic_model_t3.topics_over_time(
    tot_docs,
    df_tot['pseudo_dt'].tolist(),
    nr_bins=15,          # 15개 시간 구간
    global_tuning=True,
    evolution_tuning=True,
)

# ── 시각화: 매뉴얼 (plotly 없이 matplotlib) ───────────────────
valid_topics_t3 = ti3[ti3['Topic'] != -1]['Topic'].tolist()
topic_labels_t3 = {}
for tid in valid_topics_t3:
    top3 = [w for w, _ in topic_model_t3.get_topic(tid)[:3] if isinstance(w, str)]
    topic_labels_t3[tid] = f'T{tid}: {"/".join(top3[:2])}'

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

cmap_t = plt.cm.tab10

for ax_idx, tid in enumerate(valid_topics_t3[:8]):
    ax = axes[ax_idx]
    topic_df = topics_over_time[topics_over_time['Topic'] == tid].copy()
    topic_df['days_approx'] = (topic_df['Timestamp'] - ANCHOR).dt.days

    ax.fill_between(topic_df['days_approx'], topic_df['Frequency'],
                    alpha=0.4, color=cmap_t(ax_idx))
    ax.plot(topic_df['days_approx'], topic_df['Frequency'],
            color=cmap_t(ax_idx), linewidth=2)

    for b, label in [(30, 'D+30'), (90, 'D+90')]:
        ax.axvline(b, color='gray', linestyle=':', linewidth=1)
        ax.text(b + 1, ax.get_ylim()[1] * 0.9, label, fontsize=7, color='gray')

    ax.set_title(topic_labels_t3.get(tid, f'T{tid}'), fontsize=9, fontweight='bold')
    ax.set_xlabel('출시 후 경과 일수')
    ax.set_ylabel('빈도')

for ax in axes[len(valid_topics_t3):]:
    ax.axis('off')

plt.suptitle('BERTopic `topics_over_time` — 토픽별 시간 변화\n(회색점선=코호트 경계)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_05_topics_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-5. KeyBERT 코호트별 핵심 키워드 + UMAP 임베딩 시각화
> **TF-IDF 워드클라우드 대신** BERT 임베딩 기반 의미 키워드 추출 (KeyBERT)  
> + 각 리뷰를 2D 공간에 투영해 코호트 간 의미적 분포 차이 시각화

In [ ]:
from keybert import KeyBERT

# ── KeyBERT 초기화 (다국어 모델) ─────────────────────────────
kw_model = KeyBERT(model='paraphrase-multilingual-MiniLM-L12-v2')

# 코호트별 대표 문서 생성 (토큰 문자열 합치기)
cohort_docs_kw = {}
for cohort in COHORT_ORDER:
    texts = df_tot[df_tot['cohort'] == cohort]['token_str'].tolist()
    # 랜덤 샘플링 후 합치기 (KeyBERT는 문서 단위로 동작)
    sample = np.random.default_rng(SEED).choice(texts, size=min(200, len(texts)), replace=False)
    cohort_docs_kw[cohort] = ' '.join(sample)

# ── KeyBERT 키워드 추출 ──────────────────────────────────────
print('KeyBERT 코호트별 핵심 키워드 (의미 기반):\n')
cohort_keywords_kb = {}
for cohort, doc in cohort_docs_kw.items():
    keywords = kw_model.extract_keywords(
        doc,
        keyphrase_ngram_range=(1, 2),
        stop_words=None,
        top_n=15,
        diversity=0.5,   # MMR 다양성 조절
        use_mmr=True,
    )
    cohort_keywords_kb[cohort] = keywords
    top_str = ', '.join([f'{kw}({sc:.3f})' for kw, sc in keywords[:10]])
    print(f'[{cohort}]\n  {top_str}\n')

# ── 코호트별 키워드 비교 시각화 ──────────────────────────────
fig, axes_kb = plt.subplots(1, 3, figsize=(18, 7))

for ax, cohort in zip(axes_kb, COHORT_ORDER):
    kws = cohort_keywords_kb[cohort][:12]
    words, scores = zip(*kws)
    color = COHORT_COLORS[cohort]

    bars = ax.barh(list(words)[::-1], list(scores)[::-1],
                   color=color, alpha=0.8, edgecolor='white', linewidth=1.2)
    ax.set_title(f'{cohort}\nKeyBERT 핵심 키워드', fontsize=10, fontweight='bold',
                 color=color)
    ax.set_xlabel('의미 유사도 점수')
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('코호트별 핵심 키워드 비교 (KeyBERT — BERT 임베딩 기반)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/t3_06_keybert_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── UMAP 2D: 각 리뷰 임베딩 → 코호트별 분포 시각화 ──────────
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.1,
               metric='cosine', random_state=SEED)
emb_2d = umap_2d.fit_transform(emb2)
df_tot['umap_x'] = emb_2d[:, 0]
df_tot['umap_y'] = emb_2d[:, 1]

fig, axes_u = plt.subplots(1, 2, figsize=(16, 7))

# 코호트별 색상으로 산점도
ax_u1 = axes_u[0]
for cohort in COHORT_ORDER:
    sub = df_tot[df_tot['cohort'] == cohort]
    ax_u1.scatter(sub['umap_x'], sub['umap_y'],
                  c=COHORT_COLORS[cohort], label=cohort,
                  alpha=0.4, s=8, edgecolors='none')

# BERTopic 토픽별 중심 표시
for tid in valid_topics_t3:
    mask = df_tot['bert_topic_t3'] == tid
    if mask.sum() < 5: continue
    cx, cy = df_tot.loc[mask, 'umap_x'].mean(), df_tot.loc[mask, 'umap_y'].mean()
    lbl = topic_labels_t3.get(tid, f'T{tid}')
    ax_u1.annotate(lbl, (cx, cy), fontsize=7, ha='center',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

ax_u1.set_title('UMAP 2D — 코호트별 리뷰 의미 분포\n(BERTopic 토픽 중심 표시)',
                fontsize=11, fontweight='bold')
ax_u1.legend(markerscale=2, fontsize=9)
ax_u1.set_xlabel('UMAP 1')
ax_u1.set_ylabel('UMAP 2')

# 제품별 색상으로 산점도 (코호트 분포가 제품 특성에 따라 달라지나?)
ax_u2 = axes_u[1]
for prod in apple_prods + sam_prods:
    sub = df_tot[df_tot['product_label'] == prod]
    ax_u2.scatter(sub['umap_x'], sub['umap_y'],
                  c=BRAND_COLORS_T3[prod], label=prod,
                  alpha=0.3, s=8, edgecolors='none')

ax_u2.set_title('UMAP 2D — 제품별 리뷰 의미 분포', fontsize=11, fontweight='bold')
ax_u2.legend(markerscale=2, fontsize=8, ncol=2)
ax_u2.set_xlabel('UMAP 1')
ax_u2.set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig('output/t3_07_umap_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-6. 종합 인사이트 — 리뷰 라이프사이클 비즈니스 가치

In [ ]:
print('=' * 70)
print('Topic 3 — 리뷰 라이프사이클 종합 인사이트')
print('=' * 70)

# 전체 코호트 분포
n_early = (df_valid['cohort'] == COHORT_ORDER[0]).sum()
n_mid   = (df_valid['cohort'] == COHORT_ORDER[1]).sum()
n_late  = (df_valid['cohort'] == COHORT_ORDER[2]).sum()
n_total = n_early + n_mid + n_late
print(f'\n코호트 분포: 얼리어답터 {n_early}건({n_early/n_total*100:.1f}%) | '
      f'초기다수 {n_mid}건({n_mid/n_total*100:.1f}%) | '
      f'후기다수 {n_late}건({n_late/n_total*100:.1f}%)')

# 코호트별 평균 평점
for cohort in COHORT_ORDER:
    sub = df_valid[df_valid['cohort'] == cohort]
    print(f'  {cohort}: 평균 평점 {sub["rating"].mean():.2f} | '
          f'부정비율 {(sub["rating"]<=2).mean()*100:.1f}% | '
          f'평균도움돼요 {sub["helpfulTrueCount"].mean():.2f}')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 1. 허니문 효과는 제품마다 다르다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Galaxy Z Flip7: 얼리어답터 평점이 후기다수보다 낮음 → 역허니문
  → 출시 초기 열망 구매 후 실망 → 포폼팩터 적응 문제
• iPhone 17 Pro: 얼리어답터가 더 냉정 → 열렬한 팬보다 실용적 얼리어답터
• 허니문이 강한 제품 = 팬덤 기반 마케팅이 효과적
• 역허니문 제품 = 온보딩 개선이 초기 만족도 방어에 중요

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 2. 배송 불만은 출시 직후에 집중된다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• BERTopic topics_over_time: T1(박스/포장 불만)이 D+0~30에 집중
• 출시 초기 물량 급증 → 쿠팡 물류 과부하 → 포장 부실
• 비즈니스 제안: 신제품 출시 첫 달 물류 전담팀 배치 / 프리미엄 포장 옵션 도입

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 3. 초기 리뷰가 플랫폼 평점에 미치는 영향은 제한적
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 허니문 효과 통계 유의 제품 없음 (Mann-Whitney p > 0.05 전 제품)
• 초기 리뷰가 구조적으로 평점을 왜곡하지는 않음
• 단, helpfulness는 초기 리뷰가 높을 수 있음 → '유용한 리뷰' 알고리즘 영향 가능

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INSIGHT 4. KeyBERT로 드러난 코호트별 관심사 이동
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• 얼리어답터 → "갈아탔다", "비교", "기다렸던" (전환·기대 중심)
• 초기다수  → "화면", "카메라", "성능" (기능 실용성 중심)
• 후기다수  → "배터리", "발열", "오래쓰니" (장기 사용성 중심)
→ 출시 타이밍별로 다른 마케팅 메시지가 필요함
""")
print('분석 방법: BERTopic topics_over_time + KeyBERT (paraphrase-multilingual-MiniLM) + UMAP 2D')